# Classification binaire d'images

Notebook de suivi du projet : TP1 CNN scratch, TP2 augmentation + Dropout, TP3 transfer learning MobileNetV2.

## Phase 1.1 - Setup Colab et organisation du dataset

Objectif : configurer Colab, telecharger le dataset Kaggle, organiser les images en `train/val` par classe, puis afficher quelques exemples pour verifier le split.

In [ ]:
# === CONFIGURATION DU PROJET ===
# Dataset Kaggle : https://www.kaggle.com/datasets/chetankv/dogs-cats-images
# Ces constantes sont le seul endroit a modifier pour changer de sujet.

CLASS_A = "cat"
CLASS_B = "dog"

DATA_ROOT = "/content/data"
RAW_ROOT = f"{DATA_ROOT}/raw"
KAGGLE_DATASET = "chetankv/dogs-cats-images"
KAGGLE_ZIP = "dogs-cats-images.zip"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42
TRAIN_RATIO = 0.80

### 1.1.1 - Installation et authentification Kaggle

Dans Colab, telecharger `kaggle.json` depuis Kaggle : Settings > API > Create New Token.

In [ ]:
!pip install kaggle -q

In [ ]:
import os
from google.colab import files

uploaded = files.upload()  # selectionner kaggle.json

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("Le fichier kaggle.json doit etre selectionne.")

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
if os.path.exists(kaggle_json_path):
    os.remove(kaggle_json_path)

os.rename("kaggle.json", kaggle_json_path)
os.chmod(kaggle_json_path, 0o600)

print(f"Token Kaggle installe : {kaggle_json_path}")
!ls -la ~/.kaggle/

### 1.1.2 - Telechargement et extraction du dataset

In [ ]:
import os

os.makedirs(RAW_ROOT, exist_ok=True)
!kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_ROOT}
!unzip -q -o {RAW_ROOT}/{KAGGLE_ZIP} -d {RAW_ROOT}

print("Extraction terminee dans", RAW_ROOT)

In [ ]:
from pathlib import Path

for path in sorted(Path(RAW_ROOT).glob("**/*"))[:40]:
    print(path.relative_to(RAW_ROOT))

### 1.1.3 - Organisation en train/val

La structure finale attendue est `DATA_ROOT/train/cat`, `DATA_ROOT/train/dog`, `DATA_ROOT/val/cat`, `DATA_ROOT/val/dog`.

In [ ]:
import random
import shutil
from pathlib import Path


def is_image(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def collect_images_for_class(raw_root, class_name):
    raw_root = Path(raw_root)
    singular = class_name.lower()
    plural = f"{singular}s"

    class_dirs = [
        path for path in raw_root.glob("**/*")
        if path.is_dir() and path.name.lower() in {singular, plural}
    ]

    files = []
    for class_dir in class_dirs:
        files.extend(path for path in class_dir.rglob("*") if is_image(path))

    unique_files = sorted(set(files))
    if not unique_files:
        raise ValueError(f"Aucune image trouvee pour la classe {class_name} dans {raw_root}")

    return unique_files


files_a = collect_images_for_class(RAW_ROOT, CLASS_A)
files_b = collect_images_for_class(RAW_ROOT, CLASS_B)

print(CLASS_A, len(files_a), "images brutes")
print(CLASS_B, len(files_b), "images brutes")

In [ ]:
import os
import random
import shutil


def reset_output_dirs(data_root, classes):
    for split in ["train", "val"]:
        for cls in classes:
            path = os.path.join(data_root, split, cls)
            if os.path.exists(path):
                shutil.rmtree(path)
            os.makedirs(path, exist_ok=True)


def split_files(files, seed=SEED, train_ratio=TRAIN_RATIO):
    files = list(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    split_index = int(len(files) * train_ratio)
    return files[:split_index], files[split_index:]


def copy_split(files, split, class_name):
    target_dir = os.path.join(DATA_ROOT, split, class_name)
    for source_path in files:
        target_path = os.path.join(target_dir, source_path.name)
        if os.path.exists(target_path):
            stem, suffix = source_path.stem, source_path.suffix
            target_path = os.path.join(target_dir, f"{stem}_{abs(hash(str(source_path))) % 10**8}{suffix}")
        shutil.copy2(source_path, target_path)


reset_output_dirs(DATA_ROOT, [CLASS_A, CLASS_B])

train_a, val_a = split_files(files_a)
train_b, val_b = split_files(files_b)

copy_split(train_a, "train", CLASS_A)
copy_split(val_a, "val", CLASS_A)
copy_split(train_b, "train", CLASS_B)
copy_split(val_b, "val", CLASS_B)

print("Organisation terminee.")

### 1.1.4 - Verification des comptes

In [ ]:
counts = {}

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        path = os.path.join(DATA_ROOT, split, cls)
        image_count = len([name for name in os.listdir(path) if Path(name).suffix.lower() in IMAGE_EXTENSIONS])
        counts[(split, cls)] = image_count
        print(f"{path} : {image_count} images")

for cls in [CLASS_A, CLASS_B]:
    total = counts[("train", cls)] + counts[("val", cls)]
    train_ratio = counts[("train", cls)] / total
    print(f"{cls} : ratio train={train_ratio:.2%}, total={total}")
    assert counts[("train", cls)] >= 500, f"Pas assez d'images train pour {cls}"
    assert abs(train_ratio - TRAIN_RATIO) <= 0.01, f"Ratio train/val incorrect pour {cls}"

print("Verification des comptes OK.")

### 1.1.5 - Verification visuelle

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt


def sample_images(split, class_name, n=3):
    class_dir = Path(DATA_ROOT) / split / class_name
    images = sorted(path for path in class_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    return images[:n]


samples = [(CLASS_A, path) for path in sample_images("train", CLASS_A, 3)]
samples += [(CLASS_B, path) for path in sample_images("train", CLASS_B, 3)]

plt.figure(figsize=(12, 6))
for i, (class_name, image_path) in enumerate(samples):
    image = mpimg.imread(image_path)
    print(f"{image_path.name} | {class_name} | shape={image.shape}")

    if len(image.shape) == 3:
        assert image.shape[-1] in [1, 3, 4], f"Canaux inattendus pour {image_path}: {image.shape}"
    else:
        assert len(image.shape) == 2, f"Shape inattendue pour {image_path}: {image.shape}"

    plt.subplot(2, 3, i + 1)
    plt.imshow(image, cmap="gray" if len(image.shape) == 2 else None)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

### 1.1.6 - Edge case : seed different, ratio stable

Changer le seed modifie les fichiers selectionnes, mais pas les comptes par split sauf effet d'arrondi.

In [ ]:
def split_count_summary(files, seed):
    train_files, val_files = split_files(files, seed=seed)
    return len(train_files), len(val_files)


for cls, files in [(CLASS_A, files_a), (CLASS_B, files_b)]:
    train_42, val_42 = split_count_summary(files, seed=42)
    train_0, val_0 = split_count_summary(files, seed=0)
    print(f"{cls} | seed=42 : train={train_42}, val={val_42}")
    print(f"{cls} | seed=0  : train={train_0}, val={val_0}")
    assert abs(train_42 - train_0) <= 1
    assert abs(val_42 - val_0) <= 1

print("Verification seed OK.")

## Phase 1.2 - Preprocessing : normalisation et batching

Objectif : charger les images avec `image_dataset_from_directory`, normaliser les pixels, configurer le batching et verifier les shapes du premier batch.

In [ ]:
import os
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")

print("Dossiers train :", sorted(os.listdir(train_path)))
print("Dossiers val   :", sorted(os.listdir(val_path)))

expected_classes = sorted([CLASS_A, CLASS_B])
assert sorted(os.listdir(train_path)) == expected_classes, "Dossier parasite ou classe manquante dans train/"
assert sorted(os.listdir(val_path)) == expected_classes, "Dossier parasite ou classe manquante dans val/"

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

print("Classes detectees :", train_ds.class_names)

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

train_ds = train_ds.map(lambda images, labels: (normalization_layer(images), labels))
val_ds = val_ds.map(lambda images, labels: (normalization_layer(images), labels))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
batch_images, batch_labels = next(iter(train_ds))

print("Shape images batch :", batch_images.shape)
print("Shape labels batch :", batch_labels.shape)
print(f"Valeurs pixels : min={tf.reduce_min(batch_images).numpy():.2f}, max={tf.reduce_max(batch_images).numpy():.2f}")

assert batch_images.shape == (BATCH_SIZE, IMG_SIZE[0], IMG_SIZE[1], 3)
assert batch_labels.shape == (BATCH_SIZE, 1)
assert float(tf.reduce_min(batch_images)) >= 0.0
assert float(tf.reduce_max(batch_images)) <= 1.0

print("Pipeline dataset OK.")

Note edge case : sans `.cache()` sur `train_ds`, le pipeline relit les images depuis le disque a chaque epoch et l'entrainement devient nettement plus lent.

## Phase 1.3 - Architecture CNN from scratch

Objectif : construire et compiler un CNN simple from scratch, verifier les shapes du `model.summary()` et identifier le nombre de parametres.

### Calculs avant execution

Avec `IMG_SIZE = (128, 128)` et `padding='same'` :

- Bloc 1 apres MaxPooling : `(64, 64, 32)`
- Bloc 2 apres MaxPooling : `(32, 32, 64)`
- Bloc 3 apres MaxPooling : `(16, 16, 128)`
- Flatten : `16 * 16 * 128 = 32768` valeurs
- Dense(128) : `(32768 + 1) * 128 = 4194432` parametres

La sortie finale attendue est `(None, 1)` avec une activation `sigmoid`.

In [ ]:
from tensorflow.keras import layers, models


def build_cnn_scratch(input_shape):
    """
    CNN from scratch pour la classification binaire.
    Architecture deliberement simple : on veut voir l'overfitting, pas l'eviter.
    input_shape : tuple (H, W, C) correspondant a IMG_SIZE + nombre de canaux.
    Retourne : un modele Sequential non compile.
    """
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model


model_scratch = build_cnn_scratch(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_scratch.summary()

In [ ]:
flatten_units = (IMG_SIZE[0] // 8) * (IMG_SIZE[1] // 8) * 128
dense_128_params = (flatten_units + 1) * 128

print("Flatten attendu :", flatten_units)
print("Parametres attendus Dense(128) :", dense_128_params)

assert flatten_units == 32768
assert dense_128_params == 4_194_432
assert model_scratch.output_shape == (None, 1)
assert model_scratch.layers[-1].activation.__name__ == "sigmoid"

In [ ]:
model_scratch.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print("Modele CNN scratch compile.")

Note edge case : avec `padding='valid'`, chaque convolution reduit la hauteur et la largeur avant le pooling. Le Flatten contient donc moins d'elements et la couche `Dense(128)` a moins de parametres.

Note adversarial : la derniere couche doit rester `Dense(1, activation='sigmoid')` pour produire une probabilite compatible avec `binary_crossentropy`.

## TP2 - Augmentation + Dropout

A completer pendant la phase 2.

## TP3 - Transfer learning MobileNetV2

A completer pendant la phase 3.

## 3.4 - Tableau comparatif

A completer avec les metriques finales des trois modeles.

## 3.5 - Export ou exploration

A completer avec l'export `.tflite` et/ou une direction d'exploration.